In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("p1.ipynb")

# DATA 2201, Fall 2026

## Project 1 – IMDB Rating Analysis 

**Course:** Foundations of Data Science  
**Topic:** Pandas, Data Wrangling, EDA  
**Primary Dataset:** IMDb sample (`imdb_1000.csv`)

**Instructions**
- For each task, write your solution in the **code cell directly below** the question.
- **Do not** rename required variables — the autograder relies on exact names.
- There are both auto-graded and manually graded questions; please read the instructions and point values carefully.
- Run all cells from top to bottom before exporting for Gradescope.

**What you'll practice**
- Loading & missing value detection; type handling; de-duplication
- String & datetime derivations; binning; categorization
- Grouping, pivoting, and window operations (ranks, cumulative counts)
- Exploding lists for tidy analysis
- Joining auxiliary data
- Writing clean helper functions and saving outputs

### 🔧 Setup (Do not edit the next cell)

In [ ]:
# DO NOT EDIT THIS CELL
import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import os, urllib.request, json
import otter
grader = otter.Notebook()

### 📥 Load the dataset
If running locally, this will download `imdb_1000.csv` from a public GitHub mirror if not present.

In [ ]:
# Load the data (csv file)
df = pd.read_csv("data/imdb_1000.csv")
df.head()

### Q1 — Inspect the dataset
**Goal:** Capture basic structure information.

**Define:**
- `n_rows`: number of rows in `df` (int)
- `n_cols`: number of columns in `df` (int)
- `columns_list`: list of column names in order (list of str)

In [ ]:
n_rows = ...
n_cols = ...
columns_list = ...

n_rows, n_cols, columns_list[:5]

In [ ]:
grader.check("q1")

## Q2 — Detect Rows with Missing Values
Using the DataFrame `df` (already loaded), compute:

- `missing_counts`: a **Pandas Series** with the number of missing (`NaN`) values per column.  
- `missing_rows`: a **list** of the row indexes that contain at least one missing value.  

⚠️ Do not rename these variables — the autograder depends on them.


In [ ]:
missing_counts = ...
missing_rows = ...
missing_counts, missing_rows

In [ ]:
grader.check("q2")

### Drop Rows with Missing `content_rating`

- Using the DataFrame **df** (already loaded), drop all rows with a missing (`NaN`) value in the column **content_rating**.  
- We are not filling the missing values this time, since there are only three missing values detected in the column **content_rating**.  
- Our goal here is to focus on more advanced analysis of the IMDb data.


In [ ]:
# Drop rows where content_rating is missing
df = df.dropna(subset=["content_rating"])
df.head()

---
### 📊 Q3 — Median rating by genre
**Goal:** Compute the **median** rating per `genre` using `df_genre`, sorted descending.

**Define:**
- `median_rating_by_genre`: `pd.Series` indexed by genre with median ratings

In [ ]:
median_rating_by_genre = ...
median_rating_by_genre.head()

In [ ]:
grader.check("q3")

## Q4 — Categorize Movies by Duration
Using the DataFrame `df` (already loaded), create:

- `df_duration`: a **new DataFrame** where a new column `duration_category` is added.

Requirements:
- If `duration` < 90 → `"Short"`
- If `90 <= duration <= 150` → `"Standard"`
- If `duration` > 150 → `"Long"`
- Show the columns: `title`, `duration`, `duration_category`.


In [ ]:
df_duration = df.copy()

# Create a new categorical column based on duration
def categorize_duration(x):
    if ...:
        return ...
    ...
    ...
    else:
        return ...

df_duration["duration_category"] = ...

df_duration.head()

In [ ]:
grader.check("q4")

## Q5 — Classify Movies by Rating
Using the DataFrame `df` (already loaded), create:

- `df_classified`: a **new DataFrame** where a new column `rating_category` is added.

Requirements:
- If `star_rating` ≥ 9.0 → `"Excellent"`
- If `7.0 ≤ star_rating < 9.0` → `"Good"`
- If `star_rating < 7.0` → `"Average"`
- Show the columns: `title`, `star_rating`, `rating_category`.


In [ ]:
df_classified = df.copy()

# Function to classify ratings
def classify_rating(r):
    if r...:
        return ...
    elif r ...:
        return ...
    else:
        return ...

df_classified["rating_category"] = ...

df_classified.head()


In [ ]:
grader.check("q5")

## Q6 — Average Duration by Content Rating
Using the DataFrame `df` (already loaded), create:

- `content_duration_avg`: a **new DataFrame** that shows the **average movie duration** for each `content_rating`.

Requirements:
- Group the movies by `content_rating`.
- Compute the mean of `duration` for each content rating.
- Reset the index so the result is a clean DataFrame.
- Show the columns: `content_rating`, `avg_duration`.

In [ ]:
# Group by content rating and calculate average duration
content_duration_avg = (
    ...
    .mean()
    ....
    .rename(columns=...)
)


content_duration_avg.head()

In [ ]:
grader.check("q6")

## Q7 — Top 3 Movies per Genre (by Rating)
Using the DataFrame `df` (already loaded), create:

- `top3_per_genre`: a **new DataFrame** that shows the **top 3 highest-rated movies** within each `genre`.

Requirements:
- Use the column `star_rating` to rank movies inside each `genre`.
- Keep only the top 3 movies for each genre.
- Show the columns: `genre`, `title`, `star_rating`.
- Sort the result by `genre` (alphabetical) and `star_rating` (descending within genre).


In [ ]:
top3_per_genre = (
    df.sort_values(...)
      .groupby(...)
      .head(...)
      ...
      .reset_index(...)
)

top3_per_genre.head(15)  # show first few genres

In [ ]:
grader.check("q7")

## Q8 — Most Frequent Actor Across All Movies
Using the DataFrame `df` (already loaded), create:

- `actor_counts`: a **Series** that shows how many times each actor appears across all movies.
- `top_actor`: the name of the actor who appears most frequently.

Requirements:
- The `actors_list` column contains lists of actors (string form) → convert them to real Python lists before processing.
- Explode the column so each row corresponds to one actor per movie.
- Sort `actor_counts` in descending order.


In [ ]:
df_actor = df.copy()
df_actor["actors_list"] = df_actor["actors_list"].apply(lambda s: ast.literal_eval(s) if isinstance(s, str) else s)

# Explode to one actor per row
df_actor = ...

# Count actor appearances
actor_counts = ...

# Identify top actor
top_actor = ...

actor_counts.head(10), top_actor

In [ ]:
grader.check("q8")

## Q9 — Best (Genre, Duration-Bin) by Average Rating (Min 5 Titles)
Using the DataFrame `df` (already loaded), find the **best (genre, duration-bin) pair** by average rating.

Steps:
1. Create a new column `duration_bin` that bins movies into 10-minute intervals, e.g.:
   - 80–89 → 80s
   - 90–99 → 90s
   - 100–109 → 100s
   - etc.
   (Hint: `(duration // 10) * 10` will give you the lower bound.)
2. Group the data by `genre` and `duration_bin`.
3. Compute:
   - `avg_rating` = average `star_rating`
   - `count` = number of titles in the group
4. Filter out groups with fewer than 5 titles.
5. Identify the `(genre, duration_bin)` cell with the highest average rating.
6. Store the result in a DataFrame called `best_genre_duration` with columns:
   `genre`, `duration_bin`, `avg_rating`, `count`.


In [ ]:
df_bin = df.copy()

# Create duration bins (decades of minutes)
df_bin["duration_bin"] = ...

# Group by genre and duration_bin
grouped = (
    df_bin.groupby(...)
    .agg(avg_rating=(...), count=(...))
    .reset_index()
)

# Keep only groups with at least 5 titles
filtered = grouped[...]

# Select the best cell
best_genre_duration = filtered.sort_values(...).head(1)
best_genre_duration

In [ ]:
grader.check("q9")

## Q10 — Histogram of Movie Ratings
Using the DataFrame `df` (already loaded), create a histogram that shows the distribution of `star_rating`.

Requirements:
1. Use bins of width **0.5** (e.g., 6.0–6.5, 6.5–7.0, …).
2. Add vertical lines for:
   - **Mean rating** (red, dashed).
   - **Median rating** (blue, dashed).
3. Label the axes and add a title.


<!-- BEGIN QUESTION -->



In [ ]:
plt.figure(figsize=(8, 5))

# Plot histogram of star_rating
plt.hist(...), 
         color="skyblue", edgecolor="black", alpha=0.7)

# Mean and median
mean_rating = ...
median_rating = ...
plt.axvline(mean_rating, color="red", linestyle="--", linewidth=2, label=f"Mean = {mean_rating:.2f}")
plt.axvline(median_rating, color="blue", linestyle="--", linewidth=2, label=f"Median = {median_rating:.2f}")

# Labels and title
plt.xlabel("Star Rating")
plt.ylabel("Number of Movies")
plt.title("Distribution of IMDb Movie Ratings")
plt.legend()
plt.show()

<!-- END QUESTION -->

## Q11 — Grouping & Aggregation on Genres
Using the DataFrame `df` (already loaded):

1. Compute the **average star rating** for each `genre`.  
2. Find which `genre` has the **highest average star rating**.  
3. Create a summary table showing, for each `genre`:  
   - Average star rating (`avg_rating`)  
   - Average duration (`avg_duration`)


In [ ]:
# 1. Average star rating per genre
avg_rating_per_genre = ...

# 2. Genre with the highest average rating
best_genre = ...
best_rating = ...

# 3. Average rating and duration per genre in one table
summary_table = df.groupby(...).agg(
    avg_rating=...,
    avg_duration=...
)
summary_table

In [ ]:
grader.check("q11")

## Q12 — High-Impact Actors
Using the DataFrame `df` (already loaded):

1. Explode the `actors_list` column so that each row corresponds to one actor per movie.  
2. Filter only those actors who appear in **at least 5 movies**.  
3. For those actors, compute their **average star_rating**.  
4. Show only actors with an **average rating above 8.0**, sorted in descending order of rating.


In [ ]:
# 1. Parse and explode actors_list
df_actor = df.copy()
df_actor["actors_list"] = df_actor["actors_list"].apply(lambda s: ast.literal_eval(s) if isinstance(s, str) else s)
df_actor = ...

# 2 & 3. Group by actor: count movies and compute avg rating
actor_stats = (
    ...
    .agg(...)
    .rename(...)
)

# 4. Filter: at least 5 movies and avg rating > 8.0
high_impact_actors = ...

# Sort by average rating
high_impact_actors = ...
high_impact_actors.head()

In [ ]:
grader.check("q12")

## ✅ Congratulations — You’ve Completed Project 1!

## Submission Notes
- Make sure all **public tests** pass before exporting.  
- Before running the final cell, execute **all previous cells** and save the notebook so that your answers are recorded.  
- Finally, run the last cell to generate the ZIP file and submit it to Gradescope. The **hidden tests** will run on Gradescope after submission.


## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)